# 2:4 sparsity for the squeezed layer (features[27])

Starts from the trained dense model (92.45%), prunes the 8x8 squeezed conv to a 2:4 pattern
where in every group of 4 input channels at most 2 nonzero weights per output channel and kernel position,
fine-tunes with the mask fixed, verifies the integer data, and exports the text files for
`hardware/alpha_nm` in dense and compressed weight formats.

Requires the `nm_mask` change in `models/quant_layer.py`: the mask is applied
after quantization so pruned weights are exactly 0 in `w_int`.

In [1]:
import os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from models import *

DATA_DIR   = '../data'                                                 
MODEL_PATH = '../final_trained_model/VGG16_quant_8x8_92.45_model.pth'
RESULT_DIR = 'result/VGG16_quant_8x8_2to4_NM'
EXPORT_DIR = '../../hardware/alpha_nm/sys_data_nm'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True,
    transform=transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(), normalize]))
test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), normalize]))

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                          num_workers=2, persistent_workers=True)
testloader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                                          num_workers=2, persistent_workers=True)

device: cuda


In [2]:
def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            correct += model(data).argmax(dim=1).eq(target).sum().item()
    return 100. * correct / len(testloader.dataset)


def train_one_epoch(model, criterion, optimizer):
    model.train()
    total_loss, correct, seen = 0., 0, 0
    for data, target in trainloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += output.argmax(dim=1).eq(target).sum().item()
        seen += data.size(0)
    return total_loss / seen, 100. * correct / seen


def make_nm_mask(weight, N=2, M=4):
    # N:M mask along the input-channel dim: in each group of M consecutive input channels
    # for every output channel and kernel position keep the N largest |w - mean|,
    # the largest weights after the quantizer's mean-normalization
    oc, ic, kh, kw = weight.shape
    score = (weight - weight.mean()).abs().reshape(oc, ic // M, M, kh, kw)
    keep = score.topk(N, dim=2).indices
    return torch.zeros_like(score).scatter_(2, keep, 1.0).reshape(oc, ic, kh, kw)

In [3]:
## Load the trained dense model
model = VGG16_quant_8x8()
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)

print('Dense model accuracy: {:.2f}%'.format(evaluate(model)))

Dense model accuracy: 92.45%


In [5]:
## Prune to 2:4
N, M = 2, 4
layer = model.features[27]

with torch.no_grad():
    mask = make_nm_mask(layer.weight, N, M)
layer.register_buffer('nm_mask', mask)   # applied after quantization in forward

oc, ic, kh, kw = mask.shape
print('kept fraction:', mask.mean().item())
print('max nonzeros per group:', int(mask.reshape(oc, ic // M, M, kh, kw).sum(dim=2).max()))
print('Accuracy after 2:4 pruning (no fine-tuning): {:.2f}%'.format(evaluate(model)))

kept fraction: 0.5
max nonzeros per group: 2
Accuracy after 2:4 pruning (no fine-tuning): 85.80%


In [6]:
## Fine-tune with the mask fixed
lr = 0.005
weight_decay = 5e-4
epochs = 30

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
os.makedirs(RESULT_DIR, exist_ok=True)

best_prec = 0
for epoch in range(epochs):
    start = time.time()
    cur_lr = optimizer.param_groups[0]['lr']
    loss, train_acc = train_one_epoch(model, criterion, optimizer)
    prec = evaluate(model)
    scheduler.step()

    if prec > best_prec:
        best_prec = prec
        torch.save({'state_dict': model.state_dict(), 'best_prec': best_prec, 'epoch': epoch + 1},
                   os.path.join(RESULT_DIR, 'model_best.pth'))
    print('epoch {:2d}/{}  lr {:.5f}  loss {:.4f}  train {:.2f}%  test {:.2f}%  best {:.2f}%  ({:.0f}s)'.format(
          epoch + 1, epochs, cur_lr, loss, train_acc, prec, best_prec, time.time() - start))

epoch  1/30  lr 0.00500  loss 0.0486  train 98.43%  test 89.61%  best 89.61%  (18s)
epoch  2/30  lr 0.00499  loss 0.0581  train 98.00%  test 89.06%  best 89.61%  (11s)
epoch  3/30  lr 0.00495  loss 0.0587  train 97.96%  test 89.55%  best 89.61%  (11s)
epoch  4/30  lr 0.00488  loss 0.0553  train 98.20%  test 89.48%  best 89.61%  (12s)
epoch  5/30  lr 0.00478  loss 0.0519  train 98.24%  test 90.34%  best 90.34%  (11s)
epoch  6/30  lr 0.00467  loss 0.0500  train 98.41%  test 90.29%  best 90.34%  (11s)
epoch  7/30  lr 0.00452  loss 0.0487  train 98.40%  test 90.17%  best 90.34%  (11s)
epoch  8/30  lr 0.00436  loss 0.0429  train 98.53%  test 89.37%  best 90.34%  (11s)
epoch  9/30  lr 0.00417  loss 0.0418  train 98.64%  test 90.49%  best 90.49%  (11s)
epoch 10/30  lr 0.00397  loss 0.0351  train 98.86%  test 90.56%  best 90.56%  (11s)
epoch 11/30  lr 0.00375  loss 0.0341  train 98.92%  test 90.04%  best 90.56%  (11s)
epoch 12/30  lr 0.00352  loss 0.0300  train 99.03%  test 89.91%  best 90.56%

In [8]:
## Load the 2:4 model: prefer the saved final model, otherwise the best checkpoint
NM_FINAL_PATH = '../final_trained_model/VGG16_quant_8x8_2to4_NM_92.35_model.pth'
NM_TRAIN_PATH = os.path.join(RESULT_DIR, 'model_best.pth')

PATH = NM_FINAL_PATH if os.path.exists(NM_FINAL_PATH) else NM_TRAIN_PATH
print('Loading:', PATH)

model = VGG16_quant_8x8()
model.features[27].register_buffer('nm_mask', torch.zeros(8, 8, 3, 3))  # placeholder so the saved mask loads
checkpoint = torch.load(PATH, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)

print('mask kept fraction:', model.features[27].nm_mask.mean().item())   # should be 0.5
print('2:4 model (epoch {}): {:.2f}%'.format(checkpoint['epoch'], evaluate(model)))

Loading: ../final_trained_model/VGG16_quant_8x8_2to4_NM_92.35_model.pth
mask kept fraction: 0.5
2:4 model (epoch 30): 92.35%


In [13]:
class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []

######### Save input from 3rd layer ##########
save_8x8_in = SaveOutput()
save_8x8_out = SaveOutput()

# Extract the 3rd layer of VGG
h1 = model.features[27].register_forward_pre_hook(save_8x8_in)
h2 = model.features[29].register_forward_pre_hook(save_8x8_out)

dataiter = iter(testloader)
images, labels = next(dataiter)
images = images.to(device)
with torch.no_grad():
    out = model(images)

h1.remove()
h2.remove()

x = save_8x8_in.outputs[0][0][0] # single image
next_in = save_8x8_out.outputs[0][0][0]

#### Convert x into x_int ####
x_bit = 4

x_alpha  = model.features[27].act_alpha
x_delta = x_alpha / (2 ** x_bit - 1)

act_quant_fn = act_quantization(x_bit)
x_q = act_quant_fn(x, x_alpha)

x_int = (x_q / x_delta).round()
print("x_int shape:", x_int.shape)
print(x_int)

x_int shape: torch.Size([8, 4, 4])
tensor([[[ 2.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  4.],
         [ 0.,  0.,  0.,  3.]],

        [[ 6., 10.,  8.,  3.],
         [10., 12.,  7.,  2.],
         [ 1.,  2.,  4., 10.],
         [ 0.,  1.,  0.,  3.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  1.,  3.],
         [ 2.,  5.,  7.,  8.]],

        [[ 1.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 3.,  0.,  0.,  0.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  2.,  6.],
         [ 1.,  3.,  3.,  3.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 8.,  8.,  0.,  0.],
         [ 5.,  5.,  1.,  0.]],

        [[ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.],
         [ 0.,  0.,  0.,  0.]],

        [[ 3.,  6.,  4.,  2.],
         [ 1.,  5.,  5.,  4.],
         [ 7., 15., 1

In [14]:
weight_q = model.features[27].weight_q
w_alpha = model.features[27].weight_quant.wgt_alpha
w_bit = 4

w_delta = w_alpha / (2**(w_bit-1) - 1) 
w_int = (weight_q / w_delta).round() # Round to fix floating point error
print("w_int shape:", w_int.shape)
print(w_int)

w_int shape: torch.Size([8, 8, 3, 3])
tensor([[[[ 0., -0., -0.],
          [-0.,  7., -0.],
          [ 7.,  7.,  0.]],

         [[ 7.,  7., -7.],
          [-7.,  0., -7.],
          [ 7., -0., -7.]],

         [[-7., -7.,  7.],
          [ 0., -7.,  7.],
          [ 0.,  7., -7.]],

         [[-0., -0., -0.],
          [-7., -0., -0.],
          [ 0.,  0., -0.]],

         [[ 7., -0.,  7.],
          [ 0.,  7., -0.],
          [ 0.,  7.,  7.]],

         [[ 0., -7.,  0.],
          [-7.,  0.,  7.],
          [-7.,  0.,  7.]],

         [[ 0.,  0.,  0.],
          [-0.,  0.,  0.],
          [-0., -0.,  0.]],

         [[ 7.,  7.,  7.],
          [ 7.,  7., -7.],
          [ 7., -7.,  0.]]],


        [[[-0., -0., -7.],
          [ 0., -7.,  7.],
          [ 7., -7., -0.]],

         [[-0.,  7.,  0.],
          [-7., -7.,  0.],
          [-0., -0.,  7.]],

         [[ 7.,  7., -7.],
          [ 7.,  0.,  7.],
          [ 7.,  7.,  7.]],

         [[ 7.,  0.,  0.],
          [ 0.,  0.,

In [19]:
## check 2:4 holds on the integer weights
w_groups = (w_int != 0).reshape(8, 2, 4, 3, 3) # [oc, group, channel in group, ky, kx]
print("max nonzeros per group:", w_groups.sum(dim=2).max().item()) # must be <= 2
print("pruned weights exactly 0:", bool((w_int[model.features[27].nm_mask == 0] == 0).all())) # must be true
print("nonzero weights in layer: {} of {}".format(int((w_int != 0).sum()), w_int.numel()))

max nonzeros per group: 2
pruned weights exactly 0: True
nonzero weights in layer: 288 of 576


In [15]:
conv_int = torch.nn.Conv2d(in_channels=8, out_channels=8, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_int.weight = torch.nn.Parameter(w_int)

with torch.no_grad():
    psum_int = conv_int(x_int.unsqueeze(0))[0] # integer conv, [8, 4, 4]
    psum_recovered = psum_int * x_delta * w_delta # rescale to real units
    psum_recovered = F.relu(psum_recovered) # features[28] is a ReLU

difference = abs(psum_recovered - next_in)
print(difference.mean())
print("psum_recovered shape:", psum_recovered.shape)


tensor(5.4890e-08, device='cuda:0')
psum_recovered shape: torch.Size([8, 4, 4])


In [16]:
######### Create input.txt #########

# Pad input and flatten nij
x_padded = F.pad(x_int, (1, 1, 1, 1), value=0)
X = x_padded.reshape(8, 36)

print(X.shape)

bit_precision = 4
file = open('input.txt', 'w') #write to file
file.write('#time0row7[msb-lsb],time0row6[msb-lst],....,time0row0[msb-lst]#\n')
file.write('#time1row7[msb-lsb],time1row6[msb-lst],....,time1row0[msb-lst]#\n')
file.write('#................#\n')

for i in range(X.size(1)):  # time step
    for j in range(X.size(0)): # row #
        X_bin = '{0:04b}'.format(round(X[7-j,i].item()))
        for k in range(bit_precision):
            file.write(X_bin[k])        
        #file.write(' ')  # for visibility with blank between words, you can use
    file.write('\n')
file.close() #close file    


torch.Size([8, 36])


In [17]:
########## Create weight.txt ##########
W = w_int.reshape(8, 8, 9).permute(2, 0, 1)

bit_precision = 4
file = open('weight.txt', 'w')
file.write('#col0row7[msb-lsb],...,col0row0[msb-lsb]#\n')
file.write('#col1row7[msb-lsb],...,col1row0[msb-lsb]#\n')
file.write('#................#\n')

for kij in range(W.size(0)):
    for oc in range(W.size(1)):
        for j in range(W.size(2)):
            weight = round(W[kij, oc, 7-j].item())
            W_bin = '{0:04b}'.format(weight % 16)
            for k in range(bit_precision):
                file.write(W_bin[k])
        file.write('\n')
file.close()

########## Create weight_nm.txt (2:4 compressed, for SPARSE_NM=1) ##########
file = open('weight_nm.txt', 'w')
file.write('#2:4_compressed:col0_arrayrow3..arrayrow0,byte={00,idx,w}#\n')
file.write('#rows2g,2g+1=kept_weights_of_channels4g..4g+3,lower_ch_on_even_row#\n')
file.write('#................#\n')

for kij in range(W.size(0)):
    for oc in range(W.size(1)):
        entries = [] # (idx, w) for array rows 0..3
        for g in range(2): # group 0 = ch 0-3, group 1 = ch 4-7
            nz  = [i for i in range(4) if round(W[kij, oc, 4*g+i].item()) != 0]
            pad = [i for i in range(4) if i not in nz][:2 - len(nz)]   # < 2 nonzeros: pad with w = 0
            for i in sorted(nz + pad):
                entries.append((i, round(W[kij, oc, 4*g+i].item())))
        for i, w in reversed(entries): # row3 first (MSB) ... row0 last
            file.write('00' + '{0:02b}'.format(i) + '{0:04b}'.format(w % 16))
        file.write('\n')
file.close()

In [18]:
########## Create psum.txt ##########

# P = [oc, onij] and apply ReLU
P = F.relu(psum_int).reshape(8, 16)

bit_precision = 16
file = open('psum.txt', 'w') #write to file
file.write('#time0col7[msb-lsb],time0col6[msb-lst],....,time0col0[msb-lst]#\n')
file.write('#time1col7[msb-lsb],time1col6[msb-lst],....,time1col0[msb-lst]#\n')
file.write('#................#\n')
for i in range(P.size(1)):
    for j in range(P.size(0)):
        value = round(P[7-j, i].item())

        # Convert to signed 16-bit two's complement
        P_bin = '{0:016b}'.format(value % (2**bit_precision))

        for k in range(bit_precision):
            file.write(P_bin[k])

    file.write('\n')

file.close()
